# Notebook 04 _ Ingénierie des caractéristiques

## Objectif

Ce notebook est consacré à l'ingénierie des caractéristiques (Feature Engineering) afin d'améliorer les performances des modèles de prévision.

Les principales étapes réalisées sont :

- création des variables cibles à un horizon de prévision d'une heure ;
- génération des variables retardées (lags) ;
- création des statistiques glissantes (rolling features) ;
- ajout des variables cycliques représentant les composantes temporelles ;
- préparation du jeu de données final destiné à l'entraînement et à l'évaluation des modèles.

Le jeu de données obtenu est ensuite enregistré pour être utilisé dans le Notebook suivant consacré à la modélisation prédictive.

In [1]:
# 1. IMPORTATION DES BIBLIOTHÈQUES

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.3f}".format)

print("Bibliothèques importées.")

Bibliothèques importées.


In [2]:
# 2. DÉFINITION DES CHEMINS

PROJECT_DIR = Path.cwd().parent

PROCESSED_DATA_DIR = PROJECT_DIR / "data" / "processed"
INTERIM_DATA_DIR = PROJECT_DIR / "data" / "interim"
FIGURES_DIR = PROJECT_DIR / "figures"

PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)
INTERIM_DATA_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

INPUT_FILE = PROCESSED_DATA_DIR / "energy_preprocessed.csv"

print("Projet :", PROJECT_DIR)
print("Fichier d'entrée :", INPUT_FILE)
print("Le fichier existe :", INPUT_FILE.exists())

Projet : c:\Users\celes\Desktop\PFE
Fichier d'entrée : c:\Users\celes\Desktop\PFE\data\processed\energy_preprocessed.csv
Le fichier existe : True


# 3. Chargement et contrôle du jeu de données

Le fichier issu du Notebook 02 est chargé avec conversion directe de la
variable `Time` au format temporel.

Une copie de travail est créée afin de préserver le jeu de données
prétraité original.

In [3]:
# 3. CHARGEMENT DU JEU DE DONNÉES

energy_df = pd.read_csv(
    INPUT_FILE,
    parse_dates=["Time"]
)

feature_df = energy_df.copy()

print("Dimensions :", feature_df.shape)
print("Date minimale :", feature_df["Time"].min())
print("Date maximale :", feature_df["Time"].max())
print("Nombre de valeurs manquantes :", feature_df.isna().sum().sum())

display(feature_df.head())

Dimensions : (315648, 20)
Date minimale : 2019-01-01 00:00:00
Date maximale : 2021-12-31 23:55:00
Nombre de valeurs manquantes : 0


,Time,Season,Day_of_the_week,DHI,DNI,GHI,Wind_speed,Humidity,Temperature,PV_production,Wind_production,Electric_demand,Year,Month,Day,Hour,Minute,Weekday,Quarter,Is_weekend
0,2019-01-01 00:00:00,1,1,0.000,0.000,0.000,2.880,56.036,1.820,0,2810,22216,2019,1,1,0,0,1,1,0
1,2019-01-01 00:05:00,1,1,0.000,0.000,0.000,2.880,56.036,1.820,0,2862,22106,2019,1,1,0,5,1,1,0
2,2019-01-01 00:10:00,1,1,0.000,0.000,0.000,2.880,56.194,1.780,0,2916,22130,2019,1,1,0,10,1,1,0
3,2019-01-01 00:15:00,1,1,0.000,0.000,0.000,2.880,56.344,1.740,0,2920,22040,2019,1,1,0,15,1,1,0
4,2019-01-01 00:20:00,1,1,0.000,0.000,0.000,2.840,56.440,1.720,0,2902,21963,2019,1,1,0,20,1,1,0


In [4]:
# 4. CONTRÔLE DE L'ORDRE TEMPOREL

feature_df = (
    feature_df
    .sort_values("Time")
    .reset_index(drop=True)
)

print(
    "Série triée chronologiquement :",
    feature_df["Time"].is_monotonic_increasing
)

print(
    "Dates dupliquées :",
    feature_df["Time"].duplicated().sum()
)

print(
    "Fréquence dominante :",
    feature_df["Time"].diff().mode().iloc[0]
)

Série triée chronologiquement : True
Dates dupliquées : 0
Fréquence dominante : 0 days 00:05:00


# 5. Définition de l'horizon de prévision

La fréquence du jeu de données est de cinq minutes.

Dans une première expérimentation, l'objectif consiste à prévoir les variables
énergétiques une heure à l'avance. Une heure correspond donc à douze pas de
temps.

Les variables cibles sont :

- la production photovoltaïque ;
- la production éolienne ;
- la demande électrique.

Les valeurs futures sont créées uniquement comme cibles. Elles ne seront jamais
utilisées comme variables explicatives.

In [5]:
# 5. CRÉATION DES CIBLES À UNE HEURE

FORECAST_HORIZON_STEPS = 12  # 12 × 5 minutes = 60 minutes

target_columns = [
    "PV_production",
    "Wind_production",
    "Electric_demand"
]

for target in target_columns:
    feature_df[f"target_{target}_1h"] = (
        feature_df[target]
        .shift(-FORECAST_HORIZON_STEPS)
    )

target_names = [
    f"target_{target}_1h"
    for target in target_columns
]

display(
    feature_df[
        ["Time"] + target_columns + target_names
    ].head(15)
)

,Time,PV_production,Wind_production,Electric_demand,target_PV_production_1h,target_Wind_production_1h,target_Electric_demand_1h
0,2019-01-01 00:00:00,0,2810,22216,0.000,2554.000,21435.000
1,2019-01-01 00:05:00,0,2862,22106,0.000,2515.000,21369.000
2,2019-01-01 00:10:00,0,2916,22130,0.000,2496.000,21303.000
3,2019-01-01 00:15:00,0,2920,22040,0.000,2488.000,21247.000
4,2019-01-01 00:20:00,0,2902,21963,0.000,2486.000,21184.000
5,2019-01-01 00:25:00,0,2874,21867,0.000,2514.000,21123.000
6,2019-01-01 00:30:00,0,2845,21792,0.000,2508.000,21063.000
7,2019-01-01 00:35:00,0,2807,21731,0.000,2505.000,21009.000
8,2019-01-01 00:40:00,0,2763,21666,0.000,2520.000,20955.000
9,2019-01-01 00:45:00,0,2735,21624,0.000,2511.000,20897.000


In [6]:
# 6. VÉRIFICATION DES CIBLES

print("Horizon de prévision :", FORECAST_HORIZON_STEPS, "pas")
print("Durée de l'horizon : 1 heure")

print("\nValeurs manquantes créées dans les cibles :")
display(feature_df[target_names].isna().sum())

Horizon de prévision : 12 pas
Durée de l'horizon : 1 heure

Valeurs manquantes créées dans les cibles :


target_PV_production_1h      12
target_Wind_production_1h    12
target_Electric_demand_1h    12
dtype: int64

# 7. Création des variables temporelles cycliques

Les variables temporelles telles que l'heure, le jour de la semaine ou le mois
sont cycliques.

Par exemple, 23 h et 0 h sont deux heures très proches, bien que leurs valeurs
numériques soient éloignées. Une représentation classique sous forme d'entiers
ne permet pas aux modèles de comprendre cette proximité.

Les transformations sinus et cosinus permettent de représenter correctement
ces cycles temporels :

- cycle journalier ;
- cycle hebdomadaire ;
- cycle annuel.

La minute est intégrée à l'heure afin de conserver la résolution de cinq
minutes du jeu de données.

In [7]:
# 7. CRÉATION DES VARIABLES TEMPORELLES CYCLIQUES

# Heure décimale : 10 h 30 devient 10,5
feature_df["Hour_decimal"] = (
    feature_df["Hour"]
    + feature_df["Minute"] / 60
)

# Cycle journalier
feature_df["Hour_sin"] = np.sin(
    2 * np.pi * feature_df["Hour_decimal"] / 24
)

feature_df["Hour_cos"] = np.cos(
    2 * np.pi * feature_df["Hour_decimal"] / 24
)

# Cycle hebdomadaire
feature_df["Weekday_sin"] = np.sin(
    2 * np.pi * feature_df["Weekday"] / 7
)

feature_df["Weekday_cos"] = np.cos(
    2 * np.pi * feature_df["Weekday"] / 7
)

# Jour de l'année
feature_df["Day_of_year"] = feature_df["Time"].dt.dayofyear

# Cycle annuel
days_in_year = np.where(
    feature_df["Time"].dt.is_leap_year,
    366,
    365
)

feature_df["Year_sin"] = np.sin(
    2 * np.pi * feature_df["Day_of_year"] / days_in_year
)

feature_df["Year_cos"] = np.cos(
    2 * np.pi * feature_df["Day_of_year"] / days_in_year
)

cyclic_columns = [
    "Time",
    "Hour_decimal",
    "Hour_sin",
    "Hour_cos",
    "Weekday_sin",
    "Weekday_cos",
    "Day_of_year",
    "Year_sin",
    "Year_cos"
]

display(feature_df[cyclic_columns].head(15))

,Time,Hour_decimal,Hour_sin,Hour_cos,Weekday_sin,Weekday_cos,Day_of_year,Year_sin,Year_cos
0,2019-01-01 00:00:00,0.000,0.000,1.000,0.782,0.623,1,0.017,1.000
1,2019-01-01 00:05:00,0.083,0.022,1.000,0.782,0.623,1,0.017,1.000
2,2019-01-01 00:10:00,0.167,0.044,0.999,0.782,0.623,1,0.017,1.000
3,2019-01-01 00:15:00,0.250,0.065,0.998,0.782,0.623,1,0.017,1.000
4,2019-01-01 00:20:00,0.333,0.087,0.996,0.782,0.623,1,0.017,1.000
5,2019-01-01 00:25:00,0.417,0.109,0.994,0.782,0.623,1,0.017,1.000
6,2019-01-01 00:30:00,0.500,0.131,0.991,0.782,0.623,1,0.017,1.000
7,2019-01-01 00:35:00,0.583,0.152,0.988,0.782,0.623,1,0.017,1.000
8,2019-01-01 00:40:00,0.667,0.174,0.985,0.782,0.623,1,0.017,1.000
9,2019-01-01 00:45:00,0.750,0.195,0.981,0.782,0.623,1,0.017,1.000


In [8]:
# Vérification des bornes des variables cycliques

cyclic_summary = feature_df[
    [
        "Hour_sin",
        "Hour_cos",
        "Weekday_sin",
        "Weekday_cos",
        "Year_sin",
        "Year_cos"
    ]
].agg(["min", "max"]).T

display(cyclic_summary)

,min,max
Hour_sin,-1.000,1.000
Hour_cos,-1.000,1.000
Weekday_sin,-0.975,0.975
Weekday_cos,-0.901,1.000
Year_sin,-1.000,1.000
Year_cos,-1.000,1.000


# 8. Création des variables retardées

Les variables retardées, ou *lag features*, représentent les valeurs observées
avant l'instant courant.

Elles permettent aux modèles d'exploiter l'autocorrélation des séries
temporelles. Par exemple, la production photovoltaïque observée une heure ou
vingt-quatre heures auparavant peut apporter une information importante pour
prévoir sa valeur future.

Les retards retenus correspondent à plusieurs échelles temporelles :

- cinq minutes ;
- une heure ;
- six heures ;
- vingt-quatre heures ;
- sept jours.

Toutes ces variables utilisent exclusivement des observations passées.

In [9]:
# 8. CRÉATION DES VARIABLES RETARDÉES

lag_steps = {
    "5min": 1,
    "1h": 12,
    "6h": 72,
    "24h": 288,
    "7d": 2016
}

lag_source_columns = [
    "PV_production",
    "Wind_production",
    "Electric_demand",
    "GHI",
    "Wind_speed",
    "Temperature",
    "Humidity"
]

created_lag_columns = []

for column in lag_source_columns:
    for lag_name, lag_value in lag_steps.items():

        new_column = f"{column}_lag_{lag_name}"

        feature_df[new_column] = (
            feature_df[column]
            .shift(lag_value)
        )

        created_lag_columns.append(new_column)

print(
    "Nombre de variables retardées créées :",
    len(created_lag_columns)
)


Nombre de variables retardées créées : 35


In [10]:
display(
    feature_df.loc[
        2016:2025,
        [
            "Time",
            "PV_production",
            "PV_production_lag_5min",
            "PV_production_lag_1h",
            "PV_production_lag_6h",
            "PV_production_lag_24h",
            "PV_production_lag_7d"
        ]
    ]
)

,Time,PV_production,PV_production_lag_5min,PV_production_lag_1h,PV_production_lag_6h,PV_production_lag_24h,PV_production_lag_7d
2016,2019-01-08 00:00:00,0,0.000,75.000,75.000,0.000,0.000
2017,2019-01-08 00:05:00,0,0.000,75.000,75.000,0.000,0.000
2018,2019-01-08 00:10:00,0,0.000,75.000,75.000,0.000,0.000
2019,2019-01-08 00:15:00,0,0.000,75.000,75.000,0.000,0.000
2020,2019-01-08 00:20:00,0,0.000,75.000,75.000,0.000,0.000
2021,2019-01-08 00:25:00,0,0.000,75.000,75.000,0.000,0.000
2022,2019-01-08 00:30:00,0,0.000,75.000,75.000,0.000,0.000
2023,2019-01-08 00:35:00,0,0.000,75.000,75.000,0.000,0.000
2024,2019-01-08 00:40:00,0,0.000,75.000,75.000,0.000,0.000
2025,2019-01-08 00:45:00,0,0.000,75.000,75.000,0.000,0.000


In [11]:
# 9. CONTRÔLE DES VALEURS MANQUANTES CRÉÉES PAR LES LAGS

lag_missing_summary = (
    feature_df[created_lag_columns]
    .isna()
    .sum()
    .sort_values(ascending=False)
    .to_frame("Valeurs manquantes")
)

display(lag_missing_summary)

,Valeurs manquantes
Electric_demand_lag_7d,2016
Wind_production_lag_7d,2016
PV_production_lag_7d,2016
Temperature_lag_7d,2016
Humidity_lag_7d,2016
Wind_speed_lag_7d,2016
GHI_lag_7d,2016
Electric_demand_lag_24h,288
Wind_speed_lag_24h,288
Humidity_lag_24h,288


### Interprétation

La création des variables retardées génère des valeurs manquantes uniquement
au début de la série temporelle.

Ce comportement est normal : les premières observations ne disposent pas
encore d'un historique suffisant pour calculer certains retards.

Le retard maximal étant de sept jours, les 2016 premières observations ne
pourront pas disposer de toutes les variables retardées. Elles seront supprimées
uniquement à la fin du processus de création des caractéristiques.

Aucune imputation ne sera réalisée, car elle risquerait d'introduire des valeurs
artificielles dans la structure temporelle.

In [12]:
# 10. VÉRIFICATION DE L'ABSENCE DE FUITE TEMPORELLE

verification_index = 2500

verification = pd.DataFrame({
    "Élément": [
        "Instant courant",
        "PV courante",
        "PV retardée de 5 minutes",
        "PV réelle à t - 5 minutes",
        "PV retardée d'une heure",
        "PV réelle à t - 1 heure",
        "Cible PV à t + 1 heure",
        "PV réelle à t + 1 heure"
    ],
    "Valeur": [
        feature_df.loc[verification_index, "Time"],
        feature_df.loc[verification_index, "PV_production"],
        feature_df.loc[
            verification_index,
            "PV_production_lag_5min"
        ],
        feature_df.loc[
            verification_index - 1,
            "PV_production"
        ],
        feature_df.loc[
            verification_index,
            "PV_production_lag_1h"
        ],
        feature_df.loc[
            verification_index - 12,
            "PV_production"
        ],
        feature_df.loc[
            verification_index,
            "target_PV_production_1h"
        ],
        feature_df.loc[
            verification_index + 12,
            "PV_production"
        ]
    ]
})

display(verification)

,Élément,Valeur
0,Instant courant,2019-01-09 16:20:00
1,PV courante,691
2,PV retardée de 5 minutes,860.000
3,PV réelle à t - 5 minutes,860
4,PV retardée d'une heure,3217.000
5,PV réelle à t - 1 heure,3217
6,Cible PV à t + 1 heure,1.000
7,PV réelle à t + 1 heure,1


### Résumé des variables retardées

Les variables retardées permettent de fournir aux modèles des informations
sur l'évolution récente des séries temporelles.

Plusieurs horizons temporels ont été retenus afin de capturer les dépendances
à court terme (5 minutes, 1 heure), à moyen terme (6 heures et 24 heures)
ainsi qu'à plus long terme (7 jours).

Ces variables constitueront l'un des principaux ensembles de caractéristiques
utilisés lors de l'entraînement des modèles de prédiction.

# 11. Création des statistiques glissantes

Les statistiques glissantes résument l'évolution récente d'une série temporelle.

Elles permettent notamment de représenter :

- le niveau moyen observé durant les dernières heures ;
- la variabilité récente des conditions météorologiques ;
- la stabilité ou l'irrégularité des productions énergétiques ;
- les tendances locales précédant l'instant de prévision.

Afin d'éviter toute fuite d'information, les calculs sont précédés d'un
décalage d'une observation avec `shift(1)`. La valeur courante et les valeurs
futures ne participent donc jamais aux statistiques créées.

In [13]:
# 11. CRÉATION DES STATISTIQUES GLISSANTES

rolling_windows = {
    "1h": 12,
    "6h": 72,
    "24h": 288
}

rolling_source_columns = [
    "PV_production",
    "Wind_production",
    "Electric_demand",
    "GHI",
    "Wind_speed",
    "Temperature",
    "Humidity"
]

created_rolling_columns = []

for column in rolling_source_columns:

    # Décalage obligatoire pour exclure la valeur courante
    past_values = feature_df[column].shift(1)

    for window_name, window_size in rolling_windows.items():

        mean_column = f"{column}_rolling_mean_{window_name}"
        std_column = f"{column}_rolling_std_{window_name}"

        feature_df[mean_column] = (
            past_values
            .rolling(
                window=window_size,
                min_periods=window_size
            )
            .mean()
        )

        feature_df[std_column] = (
            past_values
            .rolling(
                window=window_size,
                min_periods=window_size
            )
            .std()
        )

        created_rolling_columns.extend([
            mean_column,
            std_column
        ])

print(
    "Nombre de statistiques glissantes créées :",
    len(created_rolling_columns)
)

Nombre de statistiques glissantes créées : 42


In [14]:
# 12. VÉRIFICATION DES STATISTIQUES GLISSANTES

rolling_check_columns = [
    "Time",
    "PV_production",
    "PV_production_rolling_mean_1h",
    "PV_production_rolling_std_1h",
    "GHI",
    "GHI_rolling_mean_1h",
    "GHI_rolling_std_1h",
    "Electric_demand",
    "Electric_demand_rolling_mean_1h"
]

display(
    feature_df.loc[
        288:300,
        rolling_check_columns
    ]
)

,Time,PV_production,PV_production_rolling_mean_1h,PV_production_rolling_std_1h,GHI,GHI_rolling_mean_1h,GHI_rolling_std_1h,Electric_demand,Electric_demand_rolling_mean_1h
288,2019-01-02 00:00:00,0,1.833,0.577,0.000,0.000,0.000,21602,22372.500
289,2019-01-02 00:05:00,0,1.667,0.778,0.000,0.000,0.000,21521,22252.250
290,2019-01-02 00:10:00,0,1.500,0.905,0.000,0.000,0.000,21515,22136.417
291,2019-01-02 00:15:00,0,1.333,0.985,0.000,0.000,0.000,21452,22030.167
292,2019-01-02 00:20:00,0,1.167,1.030,0.000,0.000,0.000,21337,21930.583
293,2019-01-02 00:25:00,0,1.000,1.044,0.000,0.000,0.000,21264,21828.500
294,2019-01-02 00:30:00,0,0.833,1.030,0.000,0.000,0.000,21161,21732.750
295,2019-01-02 00:35:00,0,0.667,0.985,0.000,0.000,0.000,21100,21637.667
296,2019-01-02 00:40:00,0,0.500,0.905,0.000,0.000,0.000,21041,21546.917
297,2019-01-02 00:45:00,0,0.333,0.778,0.000,0.000,0.000,20969,21459.000


In [15]:
# Valeurs manquantes créées par les fenêtres glissantes

rolling_missing_summary = (
    feature_df[created_rolling_columns]
    .isna()
    .sum()
    .sort_values(ascending=False)
    .to_frame("Valeurs manquantes")
)

display(rolling_missing_summary)

,Valeurs manquantes
PV_production_rolling_mean_24h,288
PV_production_rolling_std_24h,288
Wind_speed_rolling_std_24h,288
Wind_speed_rolling_mean_24h,288
GHI_rolling_std_24h,288
GHI_rolling_mean_24h,288
Electric_demand_rolling_std_24h,288
Electric_demand_rolling_mean_24h,288
Wind_production_rolling_std_24h,288
Wind_production_rolling_mean_24h,288


### Interprétation

Les statistiques glissantes décrivent le comportement récent des variables
climatiques et énergétiques à plusieurs échelles temporelles.

Les moyennes glissantes permettent de représenter les tendances locales,
tandis que les écarts-types glissants quantifient la variabilité récente.

Les valeurs manquantes observées au début de la série sont attendues, car un
historique minimal est nécessaire avant de pouvoir calculer chaque fenêtre.

Aucune imputation n'est appliquée. Les lignes ne disposant pas d'un historique
suffisant seront supprimées uniquement après la création complète des
caractéristiques.

In [16]:
# 13. CONTRÔLE MANUEL D'UNE MOYENNE GLISSANTE

verification_index = 3000
window_size = 12

manual_mean = (
    feature_df.loc[
        verification_index - window_size:
        verification_index - 1,
        "PV_production"
    ]
    .mean()
)

stored_mean = feature_df.loc[
    verification_index,
    "PV_production_rolling_mean_1h"
]

rolling_verification = pd.DataFrame({
    "Vérification": [
        "Moyenne manuelle des 12 valeurs précédentes",
        "Moyenne glissante enregistrée",
        "Différence absolue"
    ],
    "Valeur": [
        manual_mean,
        stored_mean,
        abs(manual_mean - stored_mean)
    ]
})

display(rolling_verification)

,Vérification,Valeur
0,Moyenne manuelle des 12 valeurs précédentes,5013.167
1,Moyenne glissante enregistrée,5013.167
2,Différence absolue,0.000


### Vérification de l'absence de fuite temporelle

La moyenne glissante calculée manuellement à partir des douze observations
précédentes est identique à la valeur enregistrée dans le jeu de données.

Cette vérification confirme que la valeur courante n'est pas utilisée dans les
statistiques glissantes et qu'aucune information future n'est introduite.

# 14. Création des variables de variation

Les variables de variation mesurent l'évolution récente des conditions
climatiques et énergétiques.

Elles permettent au modèle de distinguer, par exemple :

- une production photovoltaïque stable ;
- une production en forte augmentation après le lever du soleil ;
- une demande électrique en hausse ;
- une diminution rapide de la vitesse du vent.

Les différences sont calculées uniquement par rapport à des observations
passées.

In [17]:
# OPTIMISATION DU DATAFRAME POUR DATAFRAME IS HIGHLY FRAGMENTED 

feature_df = feature_df.copy()

print("Le DataFrame a été réorganisé en mémoire.")

Le DataFrame a été réorganisé en mémoire.


In [18]:
# 14. CRÉATION DES VARIABLES DE VARIATION

difference_source_columns = [
    "PV_production",
    "Wind_production",
    "Electric_demand",
    "GHI",
    "Wind_speed",
    "Temperature"
]

difference_steps = {
    "5min": 1,
    "1h": 12
}

created_difference_columns = []

for column in difference_source_columns:
    for difference_name, difference_step in difference_steps.items():

        new_column = f"{column}_diff_{difference_name}"

        feature_df[new_column] = (
            feature_df[column]
            - feature_df[column].shift(difference_step)
        )

        created_difference_columns.append(new_column)

print(
    "Nombre de variables de variation créées :",
    len(created_difference_columns)
)

Nombre de variables de variation créées : 12


### Interprétation

Les variables de différence représentent la dynamique récente des séries.

Une valeur positive indique une augmentation par rapport à l'observation
passée, tandis qu'une valeur négative indique une diminution.

Ces caractéristiques aideront les modèles à détecter les phases de montée,
de décroissance ou de stabilité des productions énergétiques et de la demande.

In [19]:
# 15. RÉSUMÉ DU FEATURE ENGINEERING

feature_summary = pd.DataFrame({
    "Élément": [
        "Observations initiales",
        "Colonnes initiales",
        "Colonnes après Feature Engineering",
        "Cibles futures",
        "Variables retardées",
        "Statistiques glissantes",
        "Variables de variation"
    ],
    "Valeur": [
        len(energy_df),
        energy_df.shape[1],
        feature_df.shape[1],
        len(target_names),
        len(created_lag_columns),
        len(created_rolling_columns),
        len(created_difference_columns)
    ]
})

display(feature_summary)

,Élément,Valeur
0,Observations initiales,315648
1,Colonnes initiales,20
2,Colonnes après Feature Engineering,120
3,Cibles futures,3
4,Variables retardées,35
5,Statistiques glissantes,42
6,Variables de variation,12


# 16. Nettoyage final du jeu de données enrichi

La création des cibles futures, des variables retardées et des statistiques
glissantes génère naturellement des valeurs manquantes au début et à la fin
de la série temporelle.

Ces valeurs ne correspondent pas à des erreurs dans les données. Elles sont
liées à l'absence d'un historique suffisant pour les premières observations
et à l'absence de valeurs futures pour les dernières observations.

Les lignes incomplètes sont donc supprimées uniquement après la création de
l'ensemble des caractéristiques.

In [20]:
# 16. NETTOYAGE FINAL DES VALEURS MANQUANTES

rows_before_cleaning = len(feature_df)
columns_before_cleaning = feature_df.shape[1]
missing_before_cleaning = feature_df.isna().sum().sum()

feature_clean = (
    feature_df
    .dropna()
    .reset_index(drop=True)
    .copy()
)

rows_after_cleaning = len(feature_clean)
removed_rows = rows_before_cleaning - rows_after_cleaning

print("Observations avant nettoyage :", rows_before_cleaning)
print("Observations après nettoyage :", rows_after_cleaning)
print("Lignes supprimées :", removed_rows)
print("Colonnes conservées :", feature_clean.shape[1])
print("Valeurs manquantes restantes :", feature_clean.isna().sum().sum())

Observations avant nettoyage : 315648
Observations après nettoyage : 313620
Lignes supprimées : 2028
Colonnes conservées : 120
Valeurs manquantes restantes : 0


In [21]:
# 17. VÉRIFICATION DE LA PÉRIODE FINALE

print("Première date conservée :", feature_clean["Time"].min())
print("Dernière date conservée :", feature_clean["Time"].max())

print(
    "Série chronologique :",
    feature_clean["Time"].is_monotonic_increasing
)

print(
    "Dates dupliquées :",
    feature_clean["Time"].duplicated().sum()
)

Première date conservée : 2019-01-08 00:00:00
Dernière date conservée : 2021-12-31 22:55:00
Série chronologique : True
Dates dupliquées : 0


### Interprétation

Le nettoyage final supprime uniquement les observations ne disposant pas d'un
historique suffisant ou d'une cible future disponible.

La grande majorité des données est conservée. La série reste triée dans l'ordre
chronologique et ne contient aucune date dupliquée.

Le jeu de données obtenu peut désormais être utilisé pour la séparation
temporelle entre les ensembles d'entraînement, de validation et de test.

# 18. Traitement des variables redondantes

Certaines variables temporelles décrivent la même information sous des formes
différentes.

Afin de limiter la redondance, les colonnes manifestement dupliquées sont
supprimées. Les variables temporelles lisibles sont néanmoins conservées à ce
stade pour faciliter l'interprétation et les contrôles ultérieurs.

La sélection définitive des variables explicatives sera réalisée dans le
Notebook 05, séparément pour chaque modèle.

In [22]:
# 18. SUPPRESSION DES COLONNES REDONDANTES

redundant_columns = [
    "Day_of_the_week",
    "Quarter"
]

existing_redundant_columns = [
    column
    for column in redundant_columns
    if column in feature_clean.columns
]

feature_clean = feature_clean.drop(
    columns=existing_redundant_columns
)

print("Colonnes supprimées :", existing_redundant_columns)
print("Dimensions après suppression :", feature_clean.shape)

Colonnes supprimées : ['Day_of_the_week', 'Quarter']
Dimensions après suppression : (313620, 118)


In [23]:
# 19. CONTRÔLES DE QUALITÉ FINAUX

quality_checks = pd.DataFrame({
    "Contrôle": [
        "Valeurs manquantes",
        "Dates dupliquées",
        "Ordre chronologique",
        "Nombre d'observations",
        "Nombre de colonnes",
        "Nombre de cibles"
    ],
    "Résultat": [
        feature_clean.isna().sum().sum(),
        feature_clean["Time"].duplicated().sum(),
        feature_clean["Time"].is_monotonic_increasing,
        feature_clean.shape[0],
        feature_clean.shape[1],
        len(target_names)
    ]
})

display(quality_checks)

,Contrôle,Résultat
0,Valeurs manquantes,0
1,Dates dupliquées,0
2,Ordre chronologique,True
3,Nombre d'observations,313620
4,Nombre de colonnes,118
5,Nombre de cibles,3


In [24]:
# Vérification des colonnes cibles

display(
    feature_clean[
        [
            "Time",
            "target_PV_production_1h",
            "target_Wind_production_1h",
            "target_Electric_demand_1h"
        ]
    ].head()
)

,Time,target_PV_production_1h,target_Wind_production_1h,target_Electric_demand_1h
0,2019-01-08 00:00:00,0.000,658.000,20600.000
1,2019-01-08 00:05:00,0.000,667.000,20514.000
2,2019-01-08 00:10:00,0.000,689.000,20423.000
3,2019-01-08 00:15:00,0.000,715.000,20425.000
4,2019-01-08 00:20:00,0.000,715.000,20396.000


In [25]:
# Vérification des valeurs infinies

numeric_data = feature_clean.select_dtypes(include="number")

infinite_count = np.isinf(
    numeric_data.to_numpy()
).sum()

print("Nombre de valeurs infinies :", infinite_count)

Nombre de valeurs infinies : 0


# 20. Sauvegarde du jeu de données enrichi

Le jeu de données final contient les variables originales, les variables
cycliques, les retards temporels, les statistiques glissantes, les variations
récentes et les trois cibles à une heure.

Il est sauvegardé dans le dossier `data/processed` afin d'être utilisé dans le
Notebook 05 pour la préparation chronologique des ensembles d'entraînement,
de validation et de test.

In [26]:
# 20. SAUVEGARDE DU DATASET FINAL

OUTPUT_FILE = (
    PROCESSED_DATA_DIR
    / "energy_features_1h.csv"
)

feature_clean.to_csv(
    OUTPUT_FILE,
    index=False
)

print("Dataset enrichi sauvegardé.")
print("Emplacement :", OUTPUT_FILE)
print("Dimensions :", feature_clean.shape)
print(
    "Taille du fichier :",
    round(OUTPUT_FILE.stat().st_size / (1024 ** 2), 2),
    "Mo"
)

Dataset enrichi sauvegardé.
Emplacement : c:\Users\celes\Desktop\PFE\data\processed\energy_features_1h.csv
Dimensions : (313620, 118)
Taille du fichier : 417.23 Mo


In [27]:
# 21. RELECTURE ET VÉRIFICATION DU FICHIER SAUVEGARDÉ

saved_check = pd.read_csv(
    OUTPUT_FILE,
    parse_dates=["Time"],
    nrows=5
)

print("Fichier existant :", OUTPUT_FILE.exists())
print("Nombre de colonnes relues :", saved_check.shape[1])

display(saved_check.head())

Fichier existant : True
Nombre de colonnes relues : 118


,Time,Season,DHI,DNI,GHI,Wind_speed,Humidity,Temperature,PV_production,Wind_production,Electric_demand,Year,Month,Day,Hour,Minute,Weekday,Is_weekend,target_PV_production_1h,target_Wind_production_1h,target_Electric_demand_1h,Hour_decimal,Hour_sin,Hour_cos,Weekday_sin,Weekday_cos,Day_of_year,Year_sin,Year_cos,PV_production_lag_5min,PV_production_lag_1h,PV_production_lag_6h,PV_production_lag_24h,PV_production_lag_7d,Wind_production_lag_5min,Wind_production_lag_1h,Wind_production_lag_6h,Wind_production_lag_24h,Wind_production_lag_7d,Electric_demand_lag_5min,Electric_demand_lag_1h,Electric_demand_lag_6h,Electric_demand_lag_24h,Electric_demand_lag_7d,GHI_lag_5min,GHI_lag_1h,GHI_lag_6h,GHI_lag_24h,GHI_lag_7d,Wind_speed_lag_5min,Wind_speed_lag_1h,Wind_speed_lag_6h,Wind_speed_lag_24h,Wind_speed_lag_7d,Temperature_lag_5min,Temperature_lag_1h,Temperature_lag_6h,Temperature_lag_24h,Temperature_lag_7d,Humidity_lag_5min,Humidity_lag_1h,Humidity_lag_6h,Humidity_lag_24h,Humidity_lag_7d,PV_production_rolling_mean_1h,PV_production_rolling_std_1h,PV_production_rolling_mean_6h,PV_production_rolling_std_6h,PV_production_rolling_mean_24h,PV_production_rolling_std_24h,Wind_production_rolling_mean_1h,Wind_production_rolling_std_1h,Wind_production_rolling_mean_6h,Wind_production_rolling_std_6h,Wind_production_rolling_mean_24h,Wind_production_rolling_std_24h,Electric_demand_rolling_mean_1h,Electric_demand_rolling_std_1h,Electric_demand_rolling_mean_6h,Electric_demand_rolling_std_6h,Electric_demand_rolling_mean_24h,Electric_demand_rolling_std_24h,GHI_rolling_mean_1h,GHI_rolling_std_1h,GHI_rolling_mean_6h,GHI_rolling_std_6h,GHI_rolling_mean_24h,GHI_rolling_std_24h,Wind_speed_rolling_mean_1h,Wind_speed_rolling_std_1h,Wind_speed_rolling_mean_6h,Wind_speed_rolling_std_6h,Wind_speed_rolling_mean_24h,Wind_speed_rolling_std_24h,Temperature_rolling_mean_1h,Temperature_rolling_std_1h,Temperature_rolling_mean_6h,Temperature_rolling_std_6h,Temperature_rolling_mean_24h,Temperature_rolling_std_24h,Humidity_rolling_mean_1h,Humidity_rolling_std_1h,Humidity_rolling_mean_6h,Humidity_rolling_std_6h,Humidity_rolling_mean_24h,Humidity_rolling_std_24h,PV_production_diff_5min,PV_production_diff_1h,Wind_production_diff_5min,Wind_production_diff_1h,Electric_demand_diff_5min,Electric_demand_diff_1h,GHI_diff_5min,GHI_diff_1h,Wind_speed_diff_5min,Wind_speed_diff_1h,Temperature_diff_5min,Temperature_diff_1h
0,2019-01-08 00:00:00,1,0.000,0.000,0.000,2.920,77.232,9.240,0,814,21490,2019,1,8,0,0,1,0,0.000,658.000,20600.000,0.000,0.000,1.000,0.782,0.623,8,0.137,0.991,0.000,75.000,75.000,0.000,0.000,824.000,916.000,1609.000,2848.000,2810.000,21596.000,22972.000,28665.000,21115.000,22216.000,0.000,0.000,0.000,0.000,0.000,2.940,2.900,2.640,3.420,2.880,9.220,9.220,9.960,7.920,1.820,77.314,77.226,76.950,80.290,56.036,68.750,21.651,73.958,8.839,1971.538,2791.243,848.333,28.637,1005.708,213.614,1282.896,729.465,22291.833,453.617,26018.667,2285.065,24016.250,2738.828,0.000,0.000,0.000,0.000,89.156,133.931,2.900,0.015,2.800,0.084,2.667,0.272,9.240,0.012,9.404,0.203,10.145,2.145,77.160,0.071,77.566,0.455,74.139,6.728,0.000,-75.000,-10.000,-102.000,-106.000,-1482.000,0.000,0.000,-0.020,0.020,0.020,0.020
1,2019-01-08 00:05:00,1,0.000,0.000,0.000,2.920,77.124,9.260,0,807,21411,2019,1,8,0,5,1,0,0.000,667.000,20514.000,0.083,0.022,1.000,0.782,0.623,8,0.137,0.991,0.000,75.000,75.000,0.000,0.000,814.000,876.000,1570.000,2891.000,2862.000,21490.000,22859.000,28613.000,21050.000,22106.000,0.000,0.000,0.000,0.000,0.000,2.920,2.900,2.640,3.400,2.880,9.240,9.260,9.940,7.920,1.820,77.232,77.034,77.040,80.290,56.036,62.500,29.194,72.917,12.412,1971.538,2791.243,839.833,20.788,994.667,202.234,1275.833,724.086,22168.333,453.345,25919.014,2324.150,24017.552,2737.532,0.000,0.000,0.000,0.000,89.156,133.931,2.902,0.016,2.804,0.083,2.666,0.269,9.242,0.010,9.394,0.193,10.150,2.142,77.161,0.071,77.570,0.451,74.129,6.721,0.000,-75.000,-7.000,-69.000,-79.000,-1448.000,0.000,0.000,0.000,0.020,0.020,0.000
2,2019-01-08 00

# Conclusion du Notebook 04

Le Feature Engineering a permis de transformer le jeu de données prétraité en
un ensemble de caractéristiques adapté à la prévision énergétique.

Les principales opérations réalisées sont les suivantes :

- définition d'un horizon de prévision d'une heure ;
- création de trois variables cibles futures ;
- représentation cyclique des variables temporelles ;
- création de variables retardées à plusieurs horizons ;
- calcul de moyennes et d'écarts-types glissants ;
- création de variables représentant les variations récentes ;
- vérification de l'absence de fuite d'information temporelle ;
- suppression des observations ne disposant pas d'un historique complet ;
- suppression de certaines variables redondantes ;
- contrôle final de la qualité des données ;
- sauvegarde d'un dataset enrichi et reproductible.

Le jeu de données final est désormais prêt pour une séparation chronologique
entre les ensembles d'entraînement, de validation et de test.

Le notebook suivant sera consacré à la préparation des données pour les modèles,
à la définition des variables explicatives et à la construction de modèles de
référence servant de base de comparaison aux modèles plus avancés.